In [1]:
import pandas as pd
import numpy as np
import os
dataset_path = "/kaggle/input/pcb-defect-dataset/pcb-defect-dataset"
print(os.listdir(dataset_path))

['data.yaml', 'val', 'test', 'train']


In [2]:
def get_path(path):
    image_path = os.path.join(dataset_path,path,"images")
    label_path = os.path.join(dataset_path,path,"labels")
    return image_path, label_path

train_img,train_lbl = get_path("train")
test_img,test_lbl = get_path("test")
val_img,val_lbl = get_path("val")

In [3]:
train_img_files = [os.path.join(train_img,f) for f in os.listdir(train_img) if f.endswith(('.jpg','.png','.jpeg'))]
test_img_files = [os.path.join(test_img,f) for f in os.listdir(test_img) if f.endswith(('.jpg','.png','.jpeg'))]
val_img_files = [os.path.join(val_img,f) for f in os.listdir(val_img) if f.endswith(('.jpg','.png','.jpeg'))]
train_lbl_files = [os.path.join(train_lbl,f) for f in os.listdir(train_lbl) if f.endswith(('.txt'))]
test_lbl_files = [os.path.join(test_lbl,f) for f in os.listdir(test_lbl) if f.endswith(('.txt'))]
val_lbl_files = [os.path.join(val_lbl,f) for f in os.listdir(val_lbl) if f.endswith(('.txt'))]

In [4]:
print(len(train_img_files),len(train_lbl_files))
print(len(test_img_files),len(test_lbl_files))
print(len(val_img_files),len(val_lbl_files))

8534 8534
1068 1068
1066 1066


In [5]:
import yaml
with open('/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset_path,"train")
data['val'] = os.path.join(dataset_path,"val")
data['test'] =  os.path.join(dataset_path,"test")

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)

In [6]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00


In [7]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

2026-01-07 17:23:20.611638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767806600.924148      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767806601.000659      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767806601.704911      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767806601.704968      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767806601.704971      17 computation_placer.cc:177] computation placer alr

Num GPUs Available: 0


2026-01-07 17:23:41.612997: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
from ultralytics import YOLO
model = YOLO("yolov8n.yaml")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=30,              
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",      
    lr0=0.003,  
    weight_decay=0.0005,
    cos_lr=True,
    patience=20,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=0.3
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.249 🚀 Python-3.12.12 torch-2.8.0+cu126 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
metrics = results  # results IS DetMetrics

with open("/kaggle/working/metrics.txt", "w") as f:
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"Precision: {metrics.box.p.mean():.4f}\n")
    f.write(f"Recall: {metrics.box.r.mean():.4f}\n")
    f.write(f"F1-score: {metrics.box.f1.mean():.4f}\n")

print("Metrics saved to metrics.txt")


In [ ]:
model = YOLO("/kaggle/working/runs/detect/train4/weights/best.pt")

model.predict(source="/kaggle/input/pcb-defect-dataset/pcb-defect-dataset/test/images",
              conf=0.25,save=True,project="/kaggle/working",name="inference_results")

In [ ]:
import shutil
import os

source_dir = "/kaggle/working/inference_results"
zip_path = "/kaggle/working/inference_results"

shutil.make_archive(zip_path, 'zip', source_dir)

print("ZIP created at:", zip_path + ".zip")
print("Size:",
      round(os.path.getsize(zip_path + ".zip") / (1024*1024), 2), "MB")
